# Install Spark and download data

In [ ]:
!apt-get update                                                                          # Update packages inside the VM
!apt-get install openjdk-8-jdk-headless -qq > /dev/null                                  # Install Java Development Kit (Required for installing Spark)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz   # Install Spark 3.5.1
!tar xzvf spark-3.5.1-bin-hadoop3.tgz                                                    # Unzip Spark 3.5.1 file
!pip install -q findspark

In [ ]:
# Set enviroment variable
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

!pip install pyspark==3.5.1

In [ ]:
# Show the processor of Google Colab
!cat /proc/cpuinfo

In [ ]:
# create a session for using Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("data_cleansing_App").getOrCreate()  #local[*] use all available processors



In [ ]:
# check Python version
import sys
sys.version_info

In [ ]:
# check Spark version
spark.version

In [ ]:
# load the data file
!wget https://file.designil.com/f/6BamyF+ -O w2_input.parquet

In [ ]:
dt_raw= spark.read.parquet('w2_input.parquet')
dt_raw

In [ ]:
dt = dt_raw.select("transaction_id", "date","product_id","price","quantity","customer_id","customer_country","total_amount")

# Data Profiling




In [ ]:
dt.show(20, truncate = False)

In [ ]:
dt.dtypes

In [ ]:
dt.printSchema()

In [ ]:
print((dt.count(), len(dt.columns)))

In [ ]:
dt.select("date","product_id","price","quantity","customer_id","total_amount").describe().show()

In [ ]:
dt.select("date","product_id","price","quantity","customer_id","total_amount").summary().show()

In [ ]:
dt.select("price").describe().show()


In [ ]:
# Find missing values
dt.summary("count").show()


In [ ]:
# Display rows that have missing values
dt.where( dt.customer_id.isNull() ).show()

# EDA - Exploratory Data Analysis

In [ ]:
dt.where(dt.price >= 600).show(5)

In [ ]:
dt.where(dt.customer_country == 'Australia').show(5)

In [ ]:
dt.where( dt.date.startswith("2024-05") ).count()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Convert Spark DataFrame to Pandas DataFrame
dt_pd = dt.toPandas()


In [ ]:
# Boxplot
sns.boxplot(x = dt_pd['price']) ;

In [ ]:
# Histogram
sns.histplot(dt_pd['price'], bins=50) ;

In [ ]:
sns.histplot(dt_pd[dt_pd['price'] < 40]['price'], bins=10) ;

In [ ]:
# Scatterplot - show relationship between two variables
sns.scatterplot(x=dt_pd.quantity, y=dt_pd.price) ;

# Data Cleansing with PySpark

In [ ]:
dt.printSchema()

In [ ]:
# Convert a string column to a datetime
from pyspark.sql import functions as f

dt_clean = dt.withColumn("date" , f.to_timestamp(dt.date, 'yyyy-MM-dd'))
dt_clean.show()

In [ ]:
dt_clean.select(f.min(dt_clean.date) , f.max(dt_clean.date)).show()

In [ ]:
dt_clean.printSchema()

In [ ]:
# Count the total transactions for the first half of January 2024
dt_clean.where( (f.dayofmonth(dt_clean.date) <= 15) & ( f.month(dt_clean.date) == 1 ) & ( f.year(dt_clean.date) == 2024 ) ).count()

In [ ]:
# Anomalies Check

# Syntactical Anomalies -> Lexical errors

dt_clean.select("customer_country").distinct().count()

In [ ]:
# Find the name of the country incorrect -> japan
dt_clean.select("customer_country").distinct().sort("customer_country").show( 40 )

In [ ]:
dt_clean.where(dt_clean['customer_country'] == 'Japane').show()

In [ ]:
# convert japane to japan
from pyspark.sql.functions import when

dt_clean_country = dt_clean.withColumn("customer_country_update", when(dt_clean['customer_country'] == 'Japane', 'Japan').otherwise(dt_clean['customer_country']))
dt_clean_country.select("customer_country_update").distinct().sort("customer_country_update").show(40)

In [ ]:
dt_clean_v2 = dt_clean_country.drop("customer_country").withColumnRenamed('customer_country_update', 'customer_country')
dt_clean_v2.show()

In [ ]:
# Semantic Anomalies -> Integrity constraints: Values are outside the acceptable range, such as a product_id being longer than it should be

dt_clean_v2.select("product_id").show(100)

In [ ]:
dt_clean_v2.select("product_id").count()

In [ ]:
# product_id equal to 5 characters

dt_clean_v2.where(dt_clean_v2["product_id"].rlike("^.{5}$")).count() / dt_clean_v2.count()

In [ ]:
dt_correct_product = dt_clean_v2.filter(dt_clean_v2["product_id"].rlike("^.{5}$"))
dt_incorrect_product = dt_clean_v2.subtract(dt_correct_product)

dt_incorrect_product.show(10)

In [ ]:
# Use .substring() function to get only the first 5 characters
dt_clean_v3 = dt_clean_v2.withColumn('product_id', f.substring('product_id', 1, 5))

In [ ]:
# check
dt_correct_product = dt_clean_v3.filter(dt_clean_v3["product_id"].rlike("^.{5}$"))
dt_incorrect_product = dt_clean_v3.subtract(dt_correct_product)

dt_incorrect_product.show(10)

In [ ]:
# Missing values


# check missing values
# col -> Spark command to select a column         sum -> Spark command to calculate the sum
from pyspark.sql.functions import col, sum

dt_nulllist = dt_clean_v3.select([ sum(col(colname).isNull().cast("int")).alias(colname) for colname in dt_clean_v3.columns ])
dt_nulllist.show()

In [ ]:
# check missing values
dt_clean_v3.summary("count").show()

In [ ]:
dt_clean_v3.where( dt_clean_v3.customer_id.isNull() ).show()

In [ ]:
dt_clean_v3.select('customer_id').distinct().show()

In [ ]:
# Replace NULL customer_id values with 00000
dt_clean_v4 = dt_clean_v3.withColumn("customer_id", when(dt_clean_v3['customer_id'].isNull(), '00000').otherwise(dt_clean_v3['customer_id']))

In [ ]:
# check missing value
dt_clean_v4.where( dt_clean_v4.customer_id.isNull() ).show()

In [ ]:
#  Outliers
sns.boxplot(x = dt_clean_v4.toPandas()['price']) ;

In [ ]:
dt_clean_v4.where( dt_clean_v4.price > 600 ).select("product_id", "price").distinct().show(truncate = False)

#  Data Cleansing with Spark SQL

In [ ]:
# Convert the data from Spark DataFrame to a TempView
dt.createOrReplaceTempView("data")
dt_sql = spark.sql("SELECT * FROM data")
dt_sql.show()

In [ ]:
dt_sql_country = spark.sql("""
SELECT distinct customer_country
FROM data
ORDER BY customer_country
""")
dt_sql_country.show(100)

In [ ]:
# convert Japane to Japan
dt_sql_transform = spark.sql("""
SELECT
    transaction_id,
    date,
    product_id,
    price,
    quantity,
    customer_id,
    CASE WHEN customer_country = 'Japane' THEN 'Japan' ELSE customer_country END AS customer_country,
    total_amount
FROM
    data
""")
dt_sql_transform.show()

In [ ]:
# check result after cleansing
dt_sql_transform.select("customer_country").distinct().sort("customer_country").show(50)

In [ ]:
# Check incorrect product_id (character>5)

dt_sql_check_productid = spark.sql("""
SELECT *
FROM data
WHERE product_id NOT RLIKE '^.{5}$'
""").show()

In [ ]:
# Use CASE WHEN to clean incorect product_id

dt_sql_productid_clean = spark.sql("""
SELECT
    transaction_id,
    CASE
      WHEN length(product_id) > 5
      THEN substr(product_id, 1, 5)
      ELSE product_id
    END AS product_id,
    price,
    quantity,
    customer_id,
    customer_country,
    total_amount

FROM
    data
""")
dt_sql_productid_clean.show()

In [ ]:
# Check the result after cleansing
dt_sql_productid_clean.filter(~dt_sql_productid_clean["product_id"].rlike("^.{5}$")).show()

# Data Export in PySpark

In [ ]:
# Save to a Parquet file
dt_clean.write.parquet("cleaned_data_output.parquet")

In [ ]:
dt2 = spark.read.parquet("cleaned_data_output.parquet")

dt2.show(10)

In [ ]:
dt_clean.write.mode("overwrite").parquet("cleaned_data_output.parquet")

In [ ]:
# Save to a CSV file
dt_clean.write.csv('cleaned_data.csv', header = True)

In [ ]:
# Excel
dt_clean.toPandas().to_excel("output.xlsx")